# Case Study: ProvSQL as a Probability Calculator

This case study uses ProvSQL as what it quietly is: an **exact, correlation-aware probability calculator that you drive in SQL**. Classic probability problems – base rates, correlated events, conditional expectation, truncated distributions – become ordinary queries, and the answers are computed *exactly* (not by sampling, unless you ask) and *correlation-aware* (the provenance circuit tracks shared events, so joint and conditional probabilities come out right without independence assumptions or hand-rolled inclusion–exclusion).

The thread tying the problems together is the conditioning operator `|` (see the chapter on probabilities): once a model is loaded, `A | B` reads as “`A` given `B`”, for discrete events, for continuous random variables, and for probabilistic aggregates alike.

The translation dictionary the calculator teaches:

| probability                   | ProvSQL / SQL               |
|-------------------------------|-----------------------------|
| an event                      | tuple(s) with `set_prob`    |
| mutually exclusive outcomes   | `repair_key`                |
| $A \wedge B$                | join (`provenance_times`)   |
| $A \vee B$                  | `UNION` (`provenance_plus`) |
| $P(A)$                      | `probability_evaluate`      |
| $P(A \mid B)$               | `A | B`                     |
| a continuous quantity         | a `random_variable`         |
| $E[X]$, $\mathrm{Var}[X]$ | `expected`, `variance`      |
| $E[X \mid C]$               | `expected(X | C)`           |

## The Scenario

An epidemiology desk at a public-health agency keeps a small probabilistic model of a screening programme and reaches for ProvSQL whenever a question is really a probability question. Five such questions follow; each is a recognisable textbook problem, and each is one query.

## Setup

*The following cells set up the database with all the content this notebook requires; run them first, ideally on a fresh database.*

======================================================================
Case Study 8: ProvSQL as a Probability Calculator

Fixture for an epidemiology desk that uses ProvSQL as an exact,
correlation-aware probability calculator.  Three small probabilistic
models, each backing one classic problem:

  screening  – the (disease, test) joint sample space (discrete Bayes)
  risk       – risk factors that share a common cause (correlation)
  cases      – per-day case contributions (aggregation)

The continuous thread builds its biomarker inline with normal(), so it
needs no table.
======================================================================

In [ ]:
-- ----------------------------------------------------------------------
-- Discrete: the joint sample space of (disease, test-positive) for a
-- screening programme, as four mutually-exclusive worlds of one group.
-- Prevalence 1%, sensitivity 90%, specificity 95%:
--   P(D=1)=.01, P(+|D=1)=.9, P(+|D=0)=.05  =>  the four joint masses below.
-- repair_key turns the four rows of group 1 into mutually-exclusive
-- possible worlds; set_prob assigns each its mass.
-- ----------------------------------------------------------------------
CREATE TABLE screening(grp int, disease boolean, positive boolean, p float);
INSERT INTO screening VALUES
  (1, true,  true,  0.009),   -- diseased, true positive
  (1, true,  false, 0.001),   -- diseased, false negative
  (1, false, true,  0.0495),  -- healthy, false positive
  (1, false, false, 0.9405);  -- healthy, true negative
SELECT repair_key('screening', 'grp');
DO $$ BEGIN PERFORM set_prob(provenance(), p) FROM screening; END $$;

In [ ]:
-- ----------------------------------------------------------------------
-- Correlation: three independent risk factors.  Two derived conditions
-- A = shared AND a1 and B = shared AND a2 both depend on `shared`, so
-- they are correlated -- the point of the second problem.
-- ----------------------------------------------------------------------
CREATE TABLE risk(id text, p float);
INSERT INTO risk VALUES ('shared', 0.5), ('a1', 0.6), ('a2', 0.7);
SELECT add_provenance('risk');
DO $$ BEGIN PERFORM set_prob(provenance(), p) FROM risk; END $$;

In [ ]:
-- ----------------------------------------------------------------------
-- Aggregation: per-day case contributions.  Each row is an uncertain
-- count that is included in its region's total with probability p.
-- ----------------------------------------------------------------------
CREATE TABLE cases(day int, region text, n int, p float);
INSERT INTO cases VALUES (1, 'North', 3, 0.5), (1, 'North', 4, 0.5), (1, 'South', 2, 0.8);
SELECT add_provenance('cases');
DO $$ BEGIN PERFORM set_prob(provenance(), p) FROM cases; END $$;

The fixture defines three tiny probabilistic models, one per problem:

- `screening(grp, disease, positive, p)` – the joint sample space of a diagnostic test, as four mutually-exclusive worlds of one group (`repair_key`): prevalence 1%, sensitivity 90%, specificity 95%.
- `risk(id, p)` – three independent risk factors, two of which will be combined into *correlated* conditions.
- `cases(day, region, n, p)` – per-day case contributions, each present with probability `p`, for the aggregation problem.

The continuous problem builds its biomarker inline with `normal`, so it needs no table.

## Problem 1: The Base-Rate Fallacy

A patient tests positive. What is the probability they have the disease? The reflex answer – the test’s sensitivity, 90% – is the base-rate fallacy. The right answer is $P(\text{disease} \mid \text{positive})$, and with a 1% prevalence it is far lower.

We name the two events by reading their provenance off the model (a scope-local, inert `provenance` fetch over the qualifying worlds), then ask the calculator for the four probabilities – including the two conditionals, written with `|`:

In [ ]:
WITH e AS (
  SELECT (SELECT provenance() FROM screening WHERE disease  GROUP BY grp) AS d,
         (SELECT provenance() FROM screening WHERE positive GROUP BY grp) AS pos)
SELECT round(probability_evaluate(d)::numeric, 4)         AS p_disease,
       round(probability_evaluate(pos)::numeric, 4)       AS p_positive,
       round(probability_evaluate(d | pos)::numeric, 4)   AS p_disease_given_pos,
       round(probability_evaluate(pos | d)::numeric, 4)   AS p_pos_given_disease
FROM e;

The result – `0.0100 | 0.0585 | 0.1538 | 0.9000` – is the whole lesson in one row. `P(positive | disease) = 0.90` recovers the sensitivity, as it must; but `P(disease | positive) = 0.1538`, not 0.90: most positives are false positives because the disease is rare. `A | B` computed $P(A \wedge B)/P(B)$ over the shared circuit – exactly Bayes’ rule, with no arithmetic on your part.

## Problem 2: Correlation That Matters

Two clinical conditions, `A` and `B`, each have a known probability. What is the probability of *at least one*? If you reach for $1-(1-P(A))(1-P(B))$, you have assumed independence – and here it is wrong, because `A` and `B` share a common cause.

We build `A = shared ∧ a1` and `B = shared ∧ a2` (both depend on the same `shared` factor), then compare ProvSQL’s exact `A ∨ B` against the independence formula:

In [ ]:
WITH t AS (
  SELECT (SELECT provenance() FROM risk WHERE id = 'shared') AS f,
         (SELECT provenance() FROM risk WHERE id = 'a1')     AS a1,
         (SELECT provenance() FROM risk WHERE id = 'a2')     AS a2)
SELECT round(probability_evaluate(provenance_times(f, a1))::numeric, 4) AS p_a,
       round(probability_evaluate(provenance_times(f, a2))::numeric, 4) AS p_b,
       round(probability_evaluate(
         provenance_plus(ARRAY[provenance_times(f, a1),
                               provenance_times(f, a2)]))::numeric, 4)  AS p_a_or_b_exact,
       round((1 - (1 - 0.3) * (1 - 0.35))::numeric, 4)                  AS p_a_or_b_naive
FROM t;

`0.3000 | 0.3500 | 0.4400 | 0.5450`: the exact answer is **0.44**, the independence formula gives **0.545** – a 24% overstatement. ProvSQL gets it right for free: because the same `shared` tuple is *one* input gate in both `A` and `B` (content-addressing makes a shared base event the same gate everywhere), the disjunction circuit performs inclusion–exclusion itself. You never wrote $P(A)+P(B)-P(A \wedge B)$.

## Problem 3: Let the Cost Optimizer Choose

`probability_evaluate` is not one algorithm but a portfolio – exact methods (independent factoring, the sieve / inclusion–exclusion, knowledge compilation, tree decomposition) and sampling estimators. Called with no method, a cost chooser picks the cheapest *exact* method that fits the circuit; you can also name one explicitly. They agree, of course:

In [ ]:
SET provsql.last_eval_method = '';
WITH t AS (
  SELECT provenance_plus(ARRAY[
           (SELECT provenance() FROM risk WHERE id = 'a1'),
           (SELECT provenance() FROM risk WHERE id = 'a2')]) AS tok)
SELECT round(probability_evaluate(tok)::numeric, 4)                   AS p_default,
       round(probability_evaluate(tok, 'independent')::numeric, 4)   AS p_independent,
       round(probability_evaluate(tok, 'monte-carlo', '200000')::numeric, 2) AS p_monte_carlo
FROM t;

All three columns read `0.88` (`a1 ∨ a2`, independent, is $1-(1-0.6)(1-0.7)$). The interesting part is *which* exact method the chooser used for the default call; it is recorded in a GUC:

In [ ]:
SHOW provsql.last_eval_method;

For this tiny independent disjunction the chooser settles on a cheap exact method rather than paying for compilation. On the correlated circuit of Problem 2 it would choose differently; the point is that the *same* call, `probability_evaluate(token)`, stays correct and stays cheap as the circuits grow – the optimizer, not you, matches method to structure.

## Problem 4: A Continuous Posterior

The calculator is not limited to discrete events. A lab biomarker is a *continuous* quantity, a `random_variable`; conditioning on it is truncation, and the conditioned distribution flows onward as a value whose moments you can take.

Take a biomarker `X ~ Normal(20, 5)` and ask for its mean, and then for its mean and variance *given* that it exceeds a referral threshold of 25 – written with the natural predicate `X | (X > 25)`:

In [ ]:
SET provsql.rv_mc_samples = 0;   -- closed-form only
WITH r AS (SELECT normal(20, 5) AS x)
SELECT round(expected(r.x)::numeric, 3)               AS e_x,
       round(expected(r.x | (r.x > 25))::numeric, 3)  AS e_given_referral,
       round(variance(r.x | (r.x > 25))::numeric, 3)  AS var_given_referral,
       (support(r.x | (r.x > 25))).lo                 AS support_lower
FROM r;

`20.000 | 27.626 | 4.977 | 25`: the unconditional mean is 20, but the posterior mean of a referred patient’s biomarker is **27.626** (the closed-form truncated-normal mean), its variance shrinks to 4.977, and its support starts at the threshold 25. `X | (X > 25)` is itself a `random_variable` – it could be stored in a column and re-queried; here we read its moments directly.

## Problem 5: Conditional Expectation of an Aggregate

Finally, the third carrier: a probabilistic *aggregate*. Each row of `cases` contributes to its region’s total only when present, so the total is a random quantity. Its expectation, and its expectation *conditioned* on an observation, are again one query.

We materialise the per-region totals (each a probabilistic `agg_token`), then ask for the expected total in the North, and the expected total *given* that the high-count day actually occurred:

In [ ]:
CREATE TABLE casesum AS SELECT region, sum(n) AS total FROM cases GROUP BY region;

SELECT cs.region,
       round(expected(cs.total)::numeric, 2) AS e_total,
       round(expected(cs.total
               | (SELECT provenance() FROM cases WHERE n = 4))::numeric, 2)
         AS e_total_given_highday
FROM casesum cs
WHERE cs.region = 'North';

`North | 3.50 | 5.50`: unconditionally the North expects $0.5\cdot 3 + 0.5\cdot 4 = 3.5$ cases; given that the `n = 4` day is confirmed, the conditional expectation rises to `5.50` (a certain 4 plus the still-uncertain 3). Conditioning composed through the aggregate exactly as it did through the discrete events and the continuous biomarker.

## What Made This Work

Five textbook problems, five one-line queries, three carriers (discrete events, a continuous random variable, a probabilistic aggregate), and one operator – `|` – meaning the same thing throughout. Two properties did the heavy lifting:

- **Exactness with a cost optimizer.** `probability_evaluate` returns the exact probability and picks the cheapest method that fits; you never chose an algorithm or accepted sampling error unless you asked for it.
- **Correlation for free.** Content-addressing makes a shared base event the same gate in every circuit, so joint, disjoint, and conditional probabilities are right without independence assumptions – the difference between 0.44 and 0.545 in Problem 2, and between 3.5 and 5.5 in Problem 5.

The model lives in the database (`add_provenance` + `set_prob` + `repair_key`), so it persists and is queried – and updated – across sessions, next to the data it is about. That is the case for ProvSQL as a probability calculator: not a toy sample space, but a real query language over real, indexed, persistent data, doing exact probability arithmetic.